<a id="top"></a>
# PanSTARRS "20 queries" using MAST's TAP service: Spatial Binning


******

## Overview

This notebook is part of a series demonstrating how to address a set of scientific questions using SQL-like Astronomical Data Query Language (ADQL) queries via a Virtual Observatory standard Table Access Protocol (TAP) service at MAST. 

This series aims to be an introduction to how complex queries can be executed using TAP (which might otherwise not be possible to specify with `astroquery`), and to be a resource for how to access MAST databases after the MAST CASJobs service is retired.

The presented queries are drawn from 20 queries for SDSS as presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014), adapted for the PanSTARRS PS1 database.

This notebook presents the subset of queries concerned with **obtaining spatially-binned counts of objects**, with both RA/Dec grids (with the local tangent plane approximation) and pre-computed hierarchical triangular mesh (HTM) ids combined with other filtering constraints.


## Learning Goals
By the end of this tutorial, you will:

- Understand how to obtain counts of objects in spatial bins using ADQL queries with TAP services.


****
### Table of Contents

1. [Introduction](#Introduction)
1. [Imports](#Imports)
1. [Connect to TAP service](#Connect-to-TAP-service)
1. [Q12: Create a gridded count of galaxies with g-r>1 and i<22.5 over -5<declination<5, and 175<right ascension<185, on a grid of 2’ arc minutes.](#Q12)
1. [Q13: Create a count of galaxies for each of the HTM triangles which satisfy a certain color cut, like 0.7g-0.5r-0.2i<1.25 and i<22.5, output it in a form adequate for visualization.](#Q13)
1. [Additional Resources](#Additional-Resources)
1. [About This Notebook](#About-this-Notebook)



*******
## Introduction

Welcome! This notebook demonstrates how to answer example scientific questions by performing SQL-like Astronomical Data Query Language (ADQL) queries accessing the PanSTARRS catalogs at MAST through a Virual Observatory Table Access Protocol (TAP) service.

This tutorial will demonstrate how to answer two scientific questions that rely on obtaining counts of objects in spatial bins (together with relevant selection criteria filtering).

These queries are taken from (or closely modeled on) the "20 queries for SDSS" presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014). Taken as a whole, this collection provides worked examples on how to leverage relational database capabilities to answer scientific questions, with the aim of providing concrete starting points and references for designing queries in other research applications, both for PanSTARRS and other large data volume missions (including Roman).

<div class="alert alert-block alert-info">
<b>Note:</b> ADQL/SQL comments follow "--". Comments are used throughout the queries to explain the purpose of specific clauses.
</div>

The workflow for this notebook consists of:
* [Main](#Main)
* [Connect to TAP service](#Connect-to-TAP-service)
* [Q12: Create a gridded count of galaxies with g-r>1 and i<22.5 over -5<declination<5, and 175<right ascension<185, on a grid of 2’ arc minutes.](#Q12)
  * [Constructing the query](#Q12:-Constructing-the-query)
  * [Inspecting & visualizing the results](#Q12:-Inspecting-&-visualizing-the-results)
* [Q13: Create a count of galaxies for each of the HTM triangles which satisfy a certain color cut, like 0.7g-0.5r-0.2i<1.25 and i<22.5, output it in a form adequate for visualization.](#Q13)
  * [Constructing the query](#Q13:-Constructing-the-query)
  * [Inspecting & visualizing the results](#Q13:-Inspecting-&-visualizing-the-results)
* [Additional Resources](#Additional-Resources)
* [About This Notebook](#About-this-Notebook)

## Imports
This tutorial makes use of the following libraries: 
- [*numpy*](https://numpy.org/) for numerical calculations
- [*pyvo*](https://pyvo.readthedocs.io) for querying the MAST catalogs via TAP
- [*matplotlib.pyplot*](https://matplotlib.org/stable/api/pyplot_summary.html#module-matplotlib.pyplot), [*PowerNorm*](https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.PowerNorm.html), [*ScalarFormatter*](https://matplotlib.org/stable/api/ticker_api.html#matplotlib.ticker.ScalarFormatter), [*MultipleLocator*](https://matplotlib.org/stable/api/ticker_api.html#matplotlib.ticker.MultipleLocator), [*make_axes_locatable*](https://matplotlib.org/stable/api/_as_gen/mpl_toolkits.axes_grid1.axes_divider.make_axes_locatable.html), [*astropy.coordinates.Angle*](https://docs.astropy.org/en/stable/api/astropy.coordinates.Angle.html), [*astropy.units*](https://docs.astropy.org/en/stable/units/ref_api.html#module-astropy.units),   for plotting data
- *time*, *datetime* to determine query duration

In [ ]:
import numpy as np
import pyvo as vo
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from matplotlib.ticker import MultipleLocator, ScalarFormatter
from astropy.coordinates import Angle
from astropy import units as u
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import datetime
import time

--------
## Connect to TAP service

For all queries, we will be connecting to the Pan-STARRS (PS1) Data release 2 (DR2) catalog. Specifically, we will use the `ps1_dr2` catalogs available from the new [postgres-backed TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs/), which offers improved performance relative to the legacy database (by factors of 100 or greater, in many cases).

See the [PS1 documentation](link_to_migration_guide_here) for information about the tables available with this new TAP service.
    
<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<b>FIX LINK TO MIGRATION GUIDE ABOVE</b>
</div>

In [ ]:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/mast_catalogs"
)

This service supports the following ADQL features, 
and will return up to 100,000 rows.

In [ ]:
TAP_service.describe()

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
The schemas for the PS1 DR2 tables are available at: https://mastdev.stsci.edu/schema_browser/#/
    <p></p>
</div>

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<i><b>**IF POSSIBLE:**</b></i>
<i>Integrete the schema browser.</i>

<i>Or else provide a simple URL link, so users can still use the schema browser even if it isn't fully integrated with the jupyter notebook?</i>
</div>

*********

## Q12

Following [Gray et al. (2002)](https://arxiv.org/abs/cs/0202014), but modifying by shifting one filter to the red (as PS1 has no `u` filter) and pushing one magnitude fainter (as PS1 is deeper than SDSS),  Query 12 poses: 

> **Create a gridded count of galaxies with g-r>1 and i<22.5 over -5<declination<5, and 175<right ascension<185, on a grid of 2’ arc minutes.**

*(Note that we also omit the "create a mask of the gridded region" portion described by Gray et al., as it is unclear how to translate this for PS1. Additionally, the included cuts help to mitigate contamination from non-galaxies or data artifacts.)*

### Q12: Constructing the query


In order to select galaxies down to such a faint magnitude limit, it is necessary to use the [`ps1_dr2.stack_object` table](link_to_migration_guide), which is based on the co-addition of all available images. This table provides all required information: object positions, Kron magnitudes (to capture galaxies' entire fluxes, versus PSF or aperture magnitudes), stack detection "primary" and "best" quality information, and PSF magnitudes (together with Kron magnitudes, separates galaxies from point sources).

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Add correct link to the migration guide in the above paragraph.
</div>

Constructing this query requires both (1) filtering to select only galaxies satsifying these color and magnitude criteria, and then (2) specifying how to bin up the results.


First, we apply selection criteria as follows:
- We apply cuts on RA and Dec to restrict to only objects in the specified region.
- We then apply the color and magnitude constraints (`gkronmag-rkronmag > 1` and `ikronmag < 22.5`).
- Finally, we apply cuts to reduce contamination and apply quality controls:
    - Require entries to both be marked as a `primarydetection`and the `bestdetection` in the `ps1_dr2.stack_object` table (`primarydetection > 0`, `bestdetection > 0`)
    - Require the sum of the `g`, `r`, `i` magnitudes to be greater than 0 (filtering out objects with missing photometry in any of these bands, denoted with values of -999)
    - Select only extended sources (using `ipsfmag > 0` and `(ipsfmag-ikronmag > 0.05)`)

Next, we define our binning grid. This requires projecting the on-sky positions onto a plane which can then be gridded as specified. 
In a local tangent plane, `RA*cos(Dec)` is a "linear" degree, while declination can be used as-is. Thus, we bin over a grid in`ramean*cos(decmean)` and `decmean` using a "group by" statement, centering the grid boundary definitions (achieved through rounding) on the center of the target region (`ramean=180, decmean=0`).

In [ ]:
# Binning factor, to accomplish 2 arcmin bins.
bfac = 30.
# Midpoint RA value
ramid = 180.

adql_query = f"""
select
   round((ramean-{ramid})*cos(radians(decmean))*{bfac},0)/{bfac}+{ramid} as racosdec,
   round(decmean*{bfac},0)/{bfac} as dec,
   count(*) as pop 
from ps1_dr2.stack_object as s
where ramean between 175 and 185
   and decmean between -5 and 5
   and primarydetection>0                      -- keep best stack objects
   and bestdetection>0
   and (gkronmag+rkronmag+ikronmag) > 0        -- remove -999 values
   and ipsfmag>0 and (ipsfmag-ikronmag > 0.05) -- extended sources (galaxies)
   and gkronmag-rkronmag > 1                   -- color constraint
   and ikronmag < 22.5                         -- faint magnitude limit
group by
   round((ramean-{ramid})*cos(radians(decmean))*{bfac},0)/{bfac}+{ramid},
   round(decmean*{bfac},0)/{bfac}
"""

We then submit (and time) this query.

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

### Q12: Inspecting & visualizing the results

This query takes about 2.5 minutes and returned 77,888 rows (spatial bins).

In [ ]:
TAP_results_q12 = job.to_table()
TAP_results_q12

We can then visualize the binned counts, by reformatting the query bin cell counts into a 2D numpy array for plotting.

We accomplish this by first creating a grid of Dec, RA*cos(Dec) (the "linear" degree in this local tangent plane) describing the bin center positions, then creating an empty 2D array and assigning the bin count values from our query results based on the bin center locations.

In [ ]:
# Create a grid of Dec, RA*cos(Dec) with spacing of 2 arcmin,
# specifying the center of each bin.

ra_range = [175, 185]
dec_range = [-5, 5]
n_bins_ra = int((ra_range[1]-ra_range[0])/(2./60.)) + 1      # Add 1 for endpoint
n_bins_dec = int((dec_range[1]-dec_range[0])/(2./60.)) + 1   # Add 1 for endpoint

dec_grid = np.linspace(dec_range[0], dec_range[1], num=n_bins_dec, endpoint=True)
racosdec_grid = np.linspace(ra_range[0], ra_range[1], num=n_bins_ra, endpoint=True)

# Create an empty bincounts array
bincounts = np.zeros((n_bins_dec, n_bins_ra), dtype=int)

# Get bincount values from query results:
for i, decg in enumerate(dec_grid):
    for j, racosdecg in enumerate(racosdec_grid):
        whm = np.where(
            (np.abs(TAP_results_q12['racosdec'] - racosdecg) < 1e-2)
            & (np.abs(TAP_results_q12['dec'] - decg) < 1e-2)
        )[0]
        if len(whm) > 0:
            bincounts[i, j] = TAP_results_q12['pop'][whm[0]]

We can now visualize these bin counts by plotting this 2D array:

In [ ]:
f, ax = plt.subplots()
f.set_size_inches(4, 4)

im = ax.imshow(
    bincounts, 
    cmap="gist_heat", 
    origin="lower", 
    interpolation="none",
    norm=PowerNorm(0.5),
    extent=(175, 185, -5, 5),
)
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(im, cax=cax)
ax.set_xlim(ax.get_xlim()[::-1])
ax.set_xlabel('RA [RA*cos(Dec); deg]')
ax.set_ylabel('Dec [deg]')
ax.set_title('Q12: Galaxy counts in 2-arcmin bins')

In total, 217,725 galaxies are included in the binned counts.

The bins in this region contain an average of only 2.8 galaxies, with a peak value at 24 counts (and a number of bins with no galaxies).

In [ ]:
# Summary statistics:
print(f"Total galaxies: {np.sum(TAP_results_q12['pop'])}")
print(f"Average galaxies per bin: {np.sum(TAP_results_q12['pop'])/len(TAP_results_q12):0.1f}")
print(f"Peak value: {np.max(TAP_results_q12['pop'])} counts")

*********

## Q13

Following Gray et al., but modifying again by shifting one filter to the red (as PS1 has no `u` filter) and pushing one magnitude fainter (PS1 is deeper than SDSS), Query 13 poses:

> **Create a count of galaxies for each of the HTM triangles which satisfy a certain color cut, like 0.7g-0.5r-0.2i<1.25 and i<22.5, output it in a form adequate for visualization.**


### Q13: Constructing the query

As with Q12, selecting faint galaxies requires using the [`ps1_dr2.stack_object` table](link_to_migration_guide). 
Again this table provides all the columns required for this query: hierarchical triangular mesh (HTM) ids, Kron magnitudes (to capture galaxies' entire fluxes, versus PSF or aperture magnitudes, PSF magnitudes (together with Kron magnitudes, separates galaxies from point sources), and stack detection "primary" and "best" quality information.

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Add correct link to the migration guide in the above paragraph.
</div>

Similar to the above example, this query will combine filtering and binning --- but in this case, the binning is greatly simplified as we can directly use the HTM ids which are available for all entries in the `stack_object` table.


We apply selection criteria to:
- Apply the color and magnitude constraints (`(0.7*gkronmag-0.5*rkronmag-0.2*ikronmag) < 1.25` and `ikronmag < 22.5`).
- Apply quality cuts and remove contamination:
    - Require entries to both be marked as a `primarydetection`and the `bestdetection` in the `ps1_dr2.stack_object` table (`primarydetection > 0`, `bestdetection > 0`)
    - Require the sum of the `g`, `r`, `i` magnitudes to be greater than 0 (filtering out objects with missing photometry in any of these bands, denoted with values of -999)
    - Select only extended sources (using `ipsfmag > 0` and `(ipsfmag-ikronmag > 0.05)`

We then bin over the set of the 8-deep HTM buckets (with each covering about 1 square degree) using a "group by" statement, retrieving the 8-deep HTM buckets by right-shifting the HTM ids by 12 (as specified by Gray et al.).

For this example, we also restrict the RA range to be within [245, 330] degrees, to ensure the query does not exceed the row limits for the MAST TAP service.

> (**Note:** If executing this query in chunks to cover the full sky, chunks over 8-deep HTM buckets should be chosen and *not* RA ranges, as RA ranges do not cleanly overlap our desired 8-deep HTM bucket tiling.)

In [ ]:
RightShift12 = np.power(4, 12)   # Divisor to go from HTM IDs to the 8-deep HTM buckets

adql_query = f"""
select (htmid/{RightShift12}) as htm_8, 
avg(ramean) as ra, 
avg(decmean) as dec,
count(*) as pop
from ps1_dr2.stack_object as s
where primarydetection>0 and bestdetection>0
  and (gkronmag+rkronmag+ikronmag) > 0                -- remove -999 values
  and (ipsfmag>0) and (ipsfmag-ikronmag > 0.05)       -- extended sources (galaxies)
  and (0.7*gkronmag-0.5*rkronmag-0.2*ikronmag) < 1.25 -- meeting this color cut
  and (ikronmag < 22.5)
and ramean between 245 and 330
group by (htmid/{RightShift12})
"""

We then submit this query to the TAP service (and capture the elapsed time).

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

### Q13: Inspecting & visualizing the results

This query takes about 2.5 minutes to run and returned 94,834 rows. 

To query the full sky PanSTARRS coverage (without RA range restrictions), it would take ~10.5 minutes to run the queries, and would result in ~400,000 total rows.

In [ ]:
TAP_results_q13 = job.to_table()
TAP_results_q13

In total, 45,530,164 galaxies are included in the binned counts (for this RA range).

In [ ]:
TAP_results_q13["pop"].sum()

We now visualize the binned counts. As these bins span a wide swath of the sky, we will display the bin counts as a scatter plot over an Aitoff projection, with each bin's point colored by the number of galaxies in that bin.

In [ ]:
# Aitoff-projection scatter plot of HTM-8 bins

f = plt.figure(figsize=(10, 5))
ax = f.add_axes([0.05, 0.05, 0.925, 0.9], projection="aitoff")
cax = f.add_axes([0.985, 0.05, 0.025, 0.9])

counts = ax.scatter(
    Angle(TAP_results_q13['ra']*u.deg).wrap_at(180 * u.deg).radian,
    Angle(TAP_results_q13['dec']*u.deg).radian,
    c=TAP_results_q13['pop'],
    lw=0,
    s=2,
    norm=PowerNorm(0.5),
    cmap="plasma",
)

ax.grid(True)
f.colorbar(counts, cax=cax)
cax.yaxis.set_major_locator(MultipleLocator(1000))
cax.yaxis.set_major_formatter(ScalarFormatter())

ax.set_xlabel('RA')
ax.set_ylabel('Dec')
ax.set_title("Q13: HTMid Gridding", pad=15)

The distribution is fairly uniform, as expected for galaxies.  Near the plane of the Milky Way (the arc that runs from middle left, up to the north (in the RA slice we have selected --- the Galactic plane also runs back down to the lower right if repeated over the full sky) there are some variations seen.  The darker regions have a lower density of objects per sky area, which is a real effect due to absorption by dust near the plane of the galaxy (which reduces galaxy brightnesses and so reduces the counts).  

The brighter regions are areas with very high stellar densities.  Those stars are often blended together with neighbors, which produces a catalog object that appears to be spatially extended.  Those objects are interpreted as galaxies and so get erroneously included in the counts.  These effects near the Milky Way would need to be corrected before using this sky density for scientific analysis (or they could simply be excluded from the sample).

We do note that one complication with this example directly using HTM binning is that **HTM bins do not have equal area on the sky**. This leads to the (spherical) triangular color pattern imprinted on the astrophysical signal on the sky (as described above). It *is* be possible to correct for the bin-area discrepancies by calculating the sky area in each bin. However, functions to perform such calculations are not available with MAST's PS1 TAP service, so determining this factor has been omitted from this tutorial example.

----------

## Additional Resources

### Table Access Protocol

- IVOA standard for RESTful web service access to tabular data
- http://www.ivoa.net/documents/TAP/

### PanSTARRS 1 DR 2

- https://outerspace.stsci.edu/display/PANSTARRS/

### Astronomical Query Data Language (2.0)

- IVOA standard for querying astronomical data in tabular format, with geometric search support
- http://www.ivoa.net/documents/latest/ADQL.html

### PyVO

- an affiliated package for [astropy](https://www.astropy.org/)
- find and retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- https://pyvo.readthedocs.io/en/latest/index.html


### Full list of MAST/TAP services
- A full list of available MAST TAP services can be found at:
- https://mast.stsci.edu/vo-tap


## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)


## About this Notebook


**Author(s):**  Rick White, Sedona Price<br>
**Keyword(s):** Tutorial, TAP, pyvo, ADQL, Pan-STARRS <br>
**First Published:** 2026-06-05<br>
**Last Updated:** 2026-06-05
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 